# DocTrust-VLM — Colab 2.2B experiment

This notebook runs the paired document-robustness experiment with **HuggingFaceTB/SmolVLM-Instruct (2.2B)** on a Colab GPU.

It performs five stages:
1. clone this private GitHub repository without printing or saving the token;
2. fetch 10 unique DocVQA-derived examples;
3. derive and **visually audit** answer-evidence boxes from dataset OCR annotations;
4. generate five paired variants and run the 2.2B model once per variant;
5. report both all-sample metrics and robustness conditioned on clean-correct examples.

**Before starting:** Runtime → Change runtime type → choose a GPU. A T4 (16 GB) is sufficient for this FP16 workflow. Colab allocation is not guaranteed.


## 1. Private GitHub access

Because the repository is private, add a Colab secret named `GITHUB_TOKEN`:

1. Create a fine-grained GitHub token with **read-only Contents** access to `doctrust-vlm`.
2. In Colab, open the key icon (**Secrets**), add `GITHUB_TOKEN`, and enable notebook access.
3. Run the cell below.

The token is passed as a temporary HTTP header. It is not printed and the repository remote does not contain it.


In [ ]:
import base64
import os
import subprocess
from pathlib import Path

from google.colab import userdata

REPO_URL = "https://github.com/TharunChougoni/doctrust-vlm.git"
REPO_DIR = Path("/content/doctrust-vlm")
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Add the read-only GITHUB_TOKEN secret in Colab first.")
header = base64.b64encode(f"x-access-token:{token}".encode()).decode()
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "-c", f"http.extraHeader=Authorization: Basic {header}",
         "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "-c",
         f"http.extraHeader=Authorization: Basic {header}", "pull", "--ff-only"],
        check=True,
    )
del token, header
os.chdir(REPO_DIR)
print("Repository ready:", Path.cwd())


## 2. Install the lightweight Colab requirements

Colab already supplies CUDA-enabled PyTorch, so this does not replace the large Torch installation.


In [ ]:
%pip install -q -r requirements-colab.txt


## 3. Verify the GPU

The 2.2B model card reports about 5 GB minimum GPU memory for one-image inference. This notebook requires at least 10 GB total VRAM to leave comfortable activation headroom.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Select Runtime → Change runtime type → GPU.")
props = torch.cuda.get_device_properties(0)
total_gb = props.total_memory / 1024**3
print("GPU:", props.name)
print(f"Total VRAM: {total_gb:.1f} GB")
if total_gb < 10:
    raise RuntimeError("This Colab workflow expects at least 10 GB total VRAM.")


## 4. Fetch 10 real DocVQA examples

The mirror supplies document images, questions, accepted answers, OCR words, character offsets and OCR boxes. The script:

- keeps unique document images using SHA-256;
- requires OCR answer-match confidence ≥ 0.95;
- maps the annotated answer character span to OCR tokens;
- unions their OCR boxes and converts coordinates from 0–1000 to 0–1;
- rejects obviously oversized boxes and distributed bare-count questions.

This automates box creation, but **does not replace visual auditing**. Source images remain untracked because DocVQA terms still apply.


In [ ]:
!python scripts/fetch_docvqa_samples.py \
  --count 10 \
  --scan 150 \
  --acknowledge-docvqa-terms


## 5. Visual audit of every evidence box

Each red rectangle must tightly cover the printed accepted answer. If a box is wrong, stop and correct that manifest row before inference. The dataset contains many letters, memoranda, tables and reports—not only `CC:` fields.


In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

ROOT = Path.cwd()
source_rows = [json.loads(line) for line in Path("data/manifests/source.jsonl").read_text().splitlines()]
cols = 2
rows_n = math.ceil(len(source_rows) / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(16, 9 * rows_n))
axes = list(axes.flat) if hasattr(axes, "flat") else [axes]
for ax, row in zip(axes, source_rows):
    image = Image.open(row["image_path"]).convert("RGB")
    x1, y1, x2, y2 = row["evidence_box"]
    draw = ImageDraw.Draw(image)
    draw.rectangle(
        (x1 * image.width, y1 * image.height, x2 * image.width, y2 * image.height),
        outline="red",
        width=max(4, image.width // 250),
    )
    ax.imshow(image)
    ax.set_title(f"{row['id']}\nQ: {row['question']}\nA: {row['answers'][0]}", fontsize=10)
    ax.axis("off")
for ax in axes[len(source_rows):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


### Audit gate

After inspecting all panels, change `BOXES_AUDITED` to `True`. This gate prevents accidental inference with unverified evidence regions.


In [ ]:
BOXES_AUDITED = False  # Change to True only after inspecting every red box above.
assert BOXES_AUDITED, "Audit every evidence box, then set BOXES_AUDITED = True."


## 6. Generate paired variants

For each source question the pipeline creates:

- clean;
- JPEG quality 35;
- Gaussian blur;
- irrelevant-region occlusion;
- answer-evidence occlusion.

All pages are capped at a 1536-pixel longest edge for stable T4 memory use.


In [ ]:
import sys
sys.path.insert(0, str(Path("src").resolve()))

from doctrust.prepare import prepare

prepared_path = prepare("configs/colab_smolvlm_2b.yaml")
prepared_rows = [json.loads(line) for line in prepared_path.read_text().splitlines()]
print("Prepared manifest:", prepared_path)
print("Source examples:", len(source_rows))
print("Prepared variants:", len(prepared_rows))


## 7. Load SmolVLM 2.2B in FP16

This is the model-download/GPU boundary. It uses batch size 1 and no runtime bitsandbytes conversion.


In [ ]:
from doctrust.config import load_config
from doctrust.modeling import DocumentVLM

config = load_config("configs/colab_smolvlm_2b.yaml")
print("Loading:", config["model"]["model_id"])
model = DocumentVLM(config["model"])
print("Model loaded on:", model.model.device)


## 8. Run resumable inference

Predictions are appended after every item. If the cell is interrupted, rerunning it skips completed IDs and continues.


In [ ]:
from doctrust.io import read_jsonl
from doctrust.run import run

predictions_path = run("configs/colab_smolvlm_2b.yaml", model=model)
predictions = read_jsonl(predictions_path)
print("Predictions:", len(predictions), "written to", predictions_path)


## 9. Evaluate honestly

`mean_anls` covers all examples. The `conditional_*` metrics include only source questions with clean ANLS ≥ 0.5. That clean-correct subset is the defensible robustness analysis: a model cannot “retain” an answer it never got right on the clean page.


In [ ]:
import json
from doctrust.evaluate import evaluate

metrics = evaluate(predictions_path)
metrics_path = Path(config["output"]["metrics"])
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")
print(json.dumps(metrics, indent=2))


## 10. Inspect every prediction

Do not report aggregate values without checking the raw outputs, especially evidence-occlusion hallucinations.


In [ ]:
import pandas as pd

prediction_table = pd.DataFrame(predictions)[
    ["source_id", "variant", "question", "answers", "prediction", "latency_seconds", "peak_vram_mb"]
]
display(prediction_table)


## 11. Download reproducibility artifacts

This exports manifests, predictions, metrics and the exact Colab configuration. Dataset images are intentionally omitted.


In [ ]:
import shutil
from google.colab import files

artifact_dir = Path("/content/doctrust-artifacts")
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
artifact_dir.mkdir()
for path in [
    Path("data/manifests/source.jsonl"),
    Path("data/manifests/source_provenance.json"),
    Path("data/manifests/prepared_colab.jsonl"),
    Path("results/colab_predictions.jsonl"),
    Path("results/colab_metrics.json"),
    Path("configs/colab_smolvlm_2b.yaml"),
]:
    shutil.copy2(path, artifact_dir / path.name)
archive = shutil.make_archive("/content/doctrust-colab-artifacts", "zip", artifact_dir)
files.download(archive)


## Interpretation checklist

- Report clean accuracy/ANLS first.
- Use conditional metrics for corruption robustness.
- A false answer after evidence occlusion is an evidence-grounding failure.
- Ten examples are a smoke benchmark, not a publication-scale result.
- Do not claim boxes are ground truth: they are OCR-derived and manually audited.
- Preserve exact model ID, configuration, manifest provenance and raw predictions.
